# Imports

In [1]:
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Authenticate

In [2]:
credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    # subscription_id="[REDACTED_SUBSCRIPTION_ID]", # your subscription id
    # workspace_name="[REDACTED_WORKSPACE]", # your workspace name
    # resource_group_name="[REDACTED_RESOURCE_GROUP]" # your resource group name
)

# Creating Data Assets from Local CSV Files

We have our churn dataset as a local csv file. Let's create a data asset from the csv file so that we can use the data asset instead of dealing with csv files

In [3]:
dataset_path = 'data/telecom_churn_v1.csv'

load_data = Data(
    path=dataset_path,
    type=AssetTypes.URI_FILE,
    description="Telecom churn dataset created from local csv file",
    name="telecom_churn_dataset_mlops",
    #version="v1" # you can also add a specific version name if you would like to. If this is not specified it will default to 1,2,3...
)

data_asset_job = ml_client.data.create_or_update(
    data=load_data
)

Uploading telecom_churn_v1.csv (< 1 MB): 100%|██████████| 84.7k/84.7k [00:00<00:00, 4.36MB/s]




We have an updated version of the churn dataset with more data points. We want to use this for our tasks instead of the older version.  

We can update the data asset we created earlier and create a new version with the updated csv file

In [5]:
dataset_path = 'data/telecom_churn_v2.csv'

load_data = Data(
    path=dataset_path,
    type=AssetTypes.URI_FILE,
    description="Telecom churn dataset version 2",
    name="telecom_churn_dataset_mlops",
    version="v2" # you can also add a specific version name if you would like to. If this is not specified it will default to 1,2,3...
)

data_asset_job = ml_client.data.create_or_update(
    data=load_data
)

Uploading telecom_churn_v2.csv (< 1 MB): 100%|██████████| 129k/129k [00:00<00:00, 5.44MB/s]




# Accessing Data Assets and Loading as Pandas Dataframes

**Accessing a specific version of the data asset**

In [6]:
data_asset = ml_client.data.get(
    name="telecom_churn_dataset_mlops", # name of the data asset
    version="1" # you can use the version name to specify the version that you wish to access
)
print(data_asset)

creation_context:
  created_at: '2026-10-17T18:42:41.283203+00:00'
  created_by: [REDACTED_NAME]
  created_by_type: CreatedByType.USER
  last_modified_at: '2026-10-17T18:42:41.292798+00:00'
description: Telecom churn dataset created from local csv file
id: /subscriptions/[REDACTED_SUBSCRIPTION_ID]/resourceGroups/[REDACTED_RESOURCE_GROUP]/providers/Microsoft.MachineLearningServices/workspaces/[REDACTED_WORKSPACE]/data/telecom_churn_dataset_mlops/versions/1
name: telecom_churn_dataset_mlops
path: azureml://subscriptions/[REDACTED_SUBSCRIPTION_ID]/resourcegroups/[REDACTED_RESOURCE_GROUP]/workspaces/[REDACTED_WORKSPACE]/datastores/workspaceblobstore/paths/LocalUpload/b3d2f0a401df3ab6ebb14fd766516a9329284163fdbafd9c0b8b0c04f6f362b0/telecom_churn_v1.csv
properties: {}
tags: {}
type: uri_file
version: '1'



In [7]:
df = pd.read_csv(data_asset.path)
print("Shape: ", df.shape)
df.head()

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.16.0) and child packages mlflow-skinny (3.15.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Att

Shape:  (2192, 11)


,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1


**Accessing the latest version of the data asset automatically**

In [9]:
data_asset = ml_client.data.get(
    name="telecom_churn_dataset_mlops", # name of the data asset
    version="v2" # you can also use label = latest to always access the latest version by default
)
print(data_asset)

creation_context:
  created_at: '2026-10-17T18:48:43.192120+00:00'
  created_by: [REDACTED_NAME]
  created_by_type: CreatedByType.USER
  last_modified_at: '2026-10-17T18:48:43.202188+00:00'
description: Telecom churn dataset version 2
id: /subscriptions/[REDACTED_SUBSCRIPTION_ID]/resourceGroups/[REDACTED_RESOURCE_GROUP]/providers/Microsoft.MachineLearningServices/workspaces/[REDACTED_WORKSPACE]/data/telecom_churn_dataset_mlops/versions/v2
name: telecom_churn_dataset_mlops
path: azureml://subscriptions/[REDACTED_SUBSCRIPTION_ID]/resourcegroups/[REDACTED_RESOURCE_GROUP]/workspaces/[REDACTED_WORKSPACE]/datastores/workspaceblobstore/paths/LocalUpload/a22bbcaa4820f3745615a9947b116f709166fd4858284f09a46ce1293ddeecc3/telecom_churn_v2.csv
properties: {}
tags: {}
type: uri_file
version: v2



In [10]:
df = pd.read_csv(data_asset.path)
print("Shape: ", df.shape)
df.head()

Shape:  (3333, 11)


,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1
